In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("TotalPoolSpendsReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")

In [ ]:
# =======================================================
# Total Pool Spends Client
# =======================================================
# Sibling of TotalJobSpendsClient and TotalAllPurposeSpendsClient.
# Differences:
#   * source DBU table: dbspend360_pool_dbu_cost
#                       (keyed (instance_pool_id, cluster_id, usage_date))
#   * target table:     dbspend360_total_pool_spends
#   * NO cloud-cost join in v1. Idle pool capacity generates zero
#     rows in system.billing.usage, and pool-tagged cloud cost is
#     not propagated through dbspend360_cloud_cost_explorer in v1
#     (plan §3.2). The cloud-cost join slot is preserved as a
#     TODO(v2) marker further down so v2 lands cleanly.
#   * NO reconciliation invariant in v1 (nothing to reconcile against;
#     reconciliation comes back in v2 once cloud_cost is populated).
#   * adds an SCD-collapse of system.compute.instance_pools so the
#     rollup denormalizes pool metadata (pool_name, node_type,
#     min_idle_instances, max_capacity,
#     idle_instance_autotermination_minutes) and the per-pool
#     delete_time → pool_deleted_at column for the §3.5 three-state
#     snapshot badge (active / deleted-visible / snapshot-missing).
#   * pool_snapshot_missing is computed BEFORE the COALESCE fallback
#     on pool_name so it captures the underlying snapshot state, not
#     the post-fallback state.
#   * Creator info is intentionally NOT denormalized.
#     system.compute.instance_pools.tags is documented as user-defined
#     tags only (excludes default tags), so the auto-applied
#     DatabricksInstancePoolCreatorId tag is not visible from the
#     system table. Creator GUID is resolved per-request in the pool
#     details modal via the REST API (§4.1, CP6).
#   * MERGE key: (instance_pool_id, cluster_id, usage_date)
class TotalPoolSpendsClient:

    TABLE_NAME = "dbspend360_total_pool_spends"

    def __init__(
        self,
        audit_table: str,
        databricks_cost_table: str,
        target_table: str,
        error_log_table: str,
        overlap_days: int,
        logger=None,
    ):
        self.audit_table = audit_table
        self.databricks_cost_table = databricks_cost_table
        self.target_table = target_table
        self.error_log_table = error_log_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("TotalPoolSpendsClient")

    def _load_pool_snapshot(self):
        # SCD-collapse system.compute.instance_pools to one row per
        # instance_pool_id carrying the most-recent metadata.
        # max_by(col, change_time) mirrors the pattern used in
        # dbspend360_all_purpose_dbu_cost_app.ipynb; the QUALIFY
        # ROW_NUMBER() OVER (... ORDER BY change_time DESC) = 1
        # alternative is holistically safer on tied change_time values
        # but breaks consistency with the existing pipeline (plan §5.5).
        #
        # Per-field notes (verified against the published
        # system.compute.instance_pools schema):
        #   * The actual column is `node_type`, NOT `node_type_id`.
        #   * `delete_time` is non-null iff the pool was deleted; the
        #     most-recent SCD row carries the delete timestamp. Carry
        #     it through as `pool_deleted_at` so the UI can render the
        #     "Deleted YYYY-MM-DD" badge from plan §3.5.
        #   * `tags['DatabricksInstancePoolCreatorId']` is NOT read
        #     here because the system table's `tags` column excludes
        #     default tags (it is documented "User-defined tags for the
        #     instance pool (does not include default tags)"), so it
        #     would return NULL on every row. Creator info is resolved
        #     per-request in the modal path via the REST API (CP6).
        return spark.sql("""
            SELECT instance_pool_id,
                   max_by(instance_pool_name,                    change_time) AS pool_name,
                   max_by(node_type,                             change_time) AS node_type,
                   max_by(min_idle_instances,                    change_time) AS min_idle_instances,
                   max_by(max_capacity,                          change_time) AS max_capacity,
                   max_by(idle_instance_autotermination_minutes, change_time)
                                                                              AS idle_instance_autotermination_minutes,
                   max_by(delete_time,                           change_time) AS pool_deleted_at
            FROM system.compute.instance_pools
            GROUP BY instance_pool_id
        """)

    def build_total_pool_spends(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(
                f"Building dbspend360_total_pool_spends for {start_dt} → {end_dt}"
            )

            dbu_df = (
                spark.table(self.databricks_cost_table)
                    .alias("dbu")
                    .filter(
                        (F.col("usage_date") >= F.lit(start_dt)) &
                        (F.col("usage_date") <= F.lit(end_dt))
                    )
            )

            if dbu_df.limit(1).count() == 0:
                self.logger.info(
                    "No pool DBU rows in this date window; nothing to roll up."
                )
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                    "SUCCESS", 0, "No DBU data in window; v1: cloud cost not yet computed",
                )
                return

            pools_df = self._load_pool_snapshot().alias("p")

            # LEFT join so DBU rows with no matching snapshot row still
            # land in the target. The snapshot-missing path is signalled
            # by pool_name being NULL post-join; the COALESCE that paints
            # the fallback name comes AFTER pool_snapshot_missing is
            # captured (plan CP3 implementation notes).
            joined = (
                dbu_df.join(
                    pools_df,
                    on=(F.col("dbu.instance_pool_id") == F.col("p.instance_pool_id")),
                    how="left",
                )
                .withColumn("pool_snapshot_missing", F.col("p.pool_name").isNull())
            )

            # ---------------------------------------------------------
            # TODO(v2): pool cloud cost join goes here.
            #
            # When the cloud-cost explorers (aws/azure/gcp) start
            # propagating the DatabricksInstancePoolId tag through to
            # dbspend360_cloud_cost_explorer (or a dedicated
            # dbspend360_pool_cloud_cost_explorer table), wire the LEFT
            # join here and project the resulting cloud_cost in place of
            # the F.lit(None) below. The total_cost expression below is
            # already written so it picks up the populated cloud_cost
            # without any code change.
            # ---------------------------------------------------------

            select_cols = [
                F.col("dbu.instance_pool_id").alias("instance_pool_id"),
                F.col("dbu.cluster_id").alias("cluster_id"),
                F.col("dbu.usage_date").alias("usage_date"),
                F.col("dbu.workspace_id").alias("workspace_id"),
                F.coalesce(
                    F.col("p.pool_name"),
                    F.concat(F.lit("Pool "), F.col("dbu.instance_pool_id")),
                ).alias("pool_name"),
                F.col("p.node_type").alias("node_type"),
                F.col("p.min_idle_instances").alias("min_idle_instances"),
                F.col("p.max_capacity").alias("max_capacity"),
                F.col("p.idle_instance_autotermination_minutes").alias(
                    "idle_instance_autotermination_minutes"
                ),
                F.col("pool_snapshot_missing"),
                F.col("p.pool_deleted_at").alias("pool_deleted_at"),
                F.col("dbu.databricks_cost").alias("databricks_cost"),
                F.lit(None).cast("double").alias("cloud_cost"),
                F.col("dbu.currency").alias("currency"),
                F.col("dbu.sku_name").alias("sku_name"),
            ]

            final_df = joined.select(*select_cols)

            final_df = (
                final_df
                .withColumn(
                    "total_cost",
                    F.coalesce(F.col("databricks_cost"), F.lit(0.0))
                    + F.coalesce(F.col("cloud_cost"), F.lit(0.0)),
                )
                .withColumn("created_at", F.current_timestamp())
                .withColumn("updated_at", F.current_timestamp())
            )
            final_df = safe_cache(final_df)

            row_count = final_df.count()

            validate_source_schema(
                final_df,
                {"instance_pool_id": "string", "cluster_id": "string",
                 "usage_date": "date", "databricks_cost": "double",
                 "cloud_cost": "double", "total_cost": "double",
                 "pool_snapshot_missing": "boolean"},
                self.target_table, self.logger,
            )
            validate_no_negative_costs(
                final_df,
                ["databricks_cost", "cloud_cost", "total_cost"],
                self.target_table, self.logger,
            )
            validate_currency_consistency(final_df, "currency", self.target_table, self.logger)

            target = DeltaTable.forName(spark, self.target_table)
            (target.alias("t")
                .merge(
                    final_df.alias("s"),
                    "t.instance_pool_id = s.instance_pool_id "
                    "AND t.cluster_id = s.cluster_id "
                    "AND t.usage_date = s.usage_date",
                )
                .whenMatchedUpdate(set={
                    "workspace_id": "s.workspace_id",
                    "pool_name": "s.pool_name",
                    "node_type": "s.node_type",
                    "min_idle_instances": "s.min_idle_instances",
                    "max_capacity": "s.max_capacity",
                    "idle_instance_autotermination_minutes":
                        "s.idle_instance_autotermination_minutes",
                    "pool_snapshot_missing": "s.pool_snapshot_missing",
                    "pool_deleted_at": "s.pool_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "cloud_cost": "s.cloud_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "updated_at": "current_timestamp()",
                })
                .whenNotMatchedInsert(values={
                    "instance_pool_id": "s.instance_pool_id",
                    "cluster_id": "s.cluster_id",
                    "usage_date": "s.usage_date",
                    "workspace_id": "s.workspace_id",
                    "pool_name": "s.pool_name",
                    "node_type": "s.node_type",
                    "min_idle_instances": "s.min_idle_instances",
                    "max_capacity": "s.max_capacity",
                    "idle_instance_autotermination_minutes":
                        "s.idle_instance_autotermination_minutes",
                    "pool_snapshot_missing": "s.pool_snapshot_missing",
                    "pool_deleted_at": "s.pool_deleted_at",
                    "databricks_cost": "s.databricks_cost",
                    "cloud_cost": "s.cloud_cost",
                    "total_cost": "s.total_cost",
                    "currency": "s.currency",
                    "sku_name": "s.sku_name",
                    "created_at": "current_timestamp()",
                    "updated_at": "current_timestamp()",
                })
                .execute()
            )

            safe_unpersist(final_df)
            get_merge_metrics(self.target_table, self.logger)

            validate_post_merge(
                self.target_table, "usage_date",
                start_dt, end_dt, row_count, self.logger,
            )

            # No reconciliation invariant in v1 (plan §5.5). Stamp the
            # audit message so the absence is visible in the audit log
            # and gets dropped in v2 when cloud_cost lands.
            log_audit_run(
                self.audit_table, self.TABLE_NAME, start_dt, end_dt,
                "SUCCESS", row_count, "v1: cloud cost not yet computed",
            )
            self.logger.info(
                f"Merged {row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class TotalPoolSpendsApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        ov_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        self.client = TotalPoolSpendsClient(
            audit_table=build_table_fqn(catalog, schema, "dbspend360_audit_log"),
            databricks_cost_table=build_table_fqn(catalog, schema, "dbspend360_pool_dbu_cost"),
            target_table=build_table_fqn(catalog, schema, "dbspend360_total_pool_spends"),
            error_log_table=build_table_fqn(catalog, schema, "dbspend360_error_log"),
            overlap_days=ov_days,
            logger=logger,
        )

    def run(self):
        self.client.build_total_pool_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = TotalPoolSpendsApp()
app.run()